# Data Labeling Algorithm

This notebook demonstrates the algorithm for labeling data in a decentralized fashion.

In [ ]:
import json
import random
from datetime import datetime
from typing import Dict, List, Optional

## User Model

Each user has:
- Basic information (id, name, email)
- Label expertise markers (accuracy per category, total labels)

In [ ]:
class User:
    def __init__(self, user_id: str, name: str, email: str):
        self.user_id = user_id
        self.name = name
        self.email = email
        self.total_labels = 0
        self.correct_labels = 0
        self.expertise_by_category = {}  # category -> accuracy score
        self.labels_by_category = {}  # category -> count
    
    def get_accuracy(self) -> float:
        """Get overall accuracy"""
        if self.total_labels == 0:
            return 0.0
        return self.correct_labels / self.total_labels
    
    def get_category_expertise(self, category: str) -> float:
        """Get expertise score for a specific category"""
        return self.expertise_by_category.get(category, 0.0)
    
    def update_label_stats(self, category: str, correct: bool):
        """Update user statistics after labeling"""
        self.total_labels += 1
        if correct:
            self.correct_labels += 1
        
        # Update category-specific stats
        if category not in self.labels_by_category:
            self.labels_by_category[category] = 0
            self.expertise_by_category[category] = 0.0
        
        self.labels_by_category[category] += 1
        
        # Recalculate category expertise (simplified)
        if correct:
            self.expertise_by_category[category] = min(
                1.0,
                self.expertise_by_category[category] + (1.0 / max(1, self.labels_by_category[category]))
            )
    
    def to_dict(self) -> Dict:
        """Convert to dictionary for JSON serialization"""
        return {
            'user_id': self.user_id,
            'name': self.name,
            'email': self.email,
            'total_labels': self.total_labels,
            'accuracy': self.get_accuracy(),
            'expertise_by_category': self.expertise_by_category,
            'labels_by_category': self.labels_by_category
        }

## Data Item Model

Represents an item to be labeled

In [ ]:
class DataItem:
    def __init__(self, item_id: str, content: str, content_type: str, 
                 categories: List[str], is_binary: bool = False):
        self.item_id = item_id
        self.content = content  # File path or URL
        self.content_type = content_type  # 'image', 'text', etc.
        self.categories = categories  # Possible categories
        self.is_binary = is_binary  # True for binary classification
        self.labels = []  # List of labels from different users
        self.consensus_label = None
        self.confidence = 0.0
    
    def add_label(self, user_id: str, label: str, user_expertise: float = 0.5):
        """Add a label from a user"""
        self.labels.append({
            'user_id': user_id,
            'label': label,
            'expertise': user_expertise,
            'timestamp': datetime.now().isoformat()
        })
        self._calculate_consensus()
    
    def _calculate_consensus(self):
        """Calculate consensus label using weighted voting"""
        if not self.labels:
            return
        
        # Weighted voting based on user expertise
        label_scores = {}
        total_weight = 0.0
        
        for label_data in self.labels:
            label = label_data['label']
            expertise = label_data['expertise']
            weight = max(0.1, expertise)  # Minimum weight of 0.1
            
            if label not in label_scores:
                label_scores[label] = 0.0
            label_scores[label] += weight
            total_weight += weight
        
        # Find consensus
        if label_scores:
            self.consensus_label = max(label_scores, key=label_scores.get)
            self.confidence = label_scores[self.consensus_label] / total_weight
    
    def to_dict(self) -> Dict:
        """Convert to dictionary for JSON serialization"""
        return {
            'item_id': self.item_id,
            'content': self.content,
            'content_type': self.content_type,
            'categories': self.categories,
            'is_binary': self.is_binary,
            'labels_count': len(self.labels),
            'consensus_label': self.consensus_label,
            'confidence': self.confidence
        }

## Labeling System

Manages the labeling workflow

In [ ]:
class LabelingSystem:
    def __init__(self):
        self.users = {}  # user_id -> User
        self.data_items = {}  # item_id -> DataItem
        self.user_labels = {}  # user_id -> [item_ids]
    
    def add_user(self, user: User):
        """Add a user to the system"""
        self.users[user.user_id] = user
        if user.user_id not in self.user_labels:
            self.user_labels[user.user_id] = []
    
    def add_data_item(self, item: DataItem):
        """Add a data item to be labeled"""
        self.data_items[item.item_id] = item
    
    def get_next_item_for_user(self, user_id: str) -> Optional[DataItem]:
        """Get the next item for a user to label"""
        if user_id not in self.users:
            return None
        
        # Find items not yet labeled by this user
        labeled_items = set(self.user_labels.get(user_id, []))
        
        unlabeled = [
            item for item_id, item in self.data_items.items()
            if item_id not in labeled_items
        ]
        
        if not unlabeled:
            return None
        
        # Prioritize items with fewer labels or lower confidence
        unlabeled.sort(key=lambda x: (len(x.labels), x.confidence))
        return unlabeled[0]
    
    def submit_label(self, user_id: str, item_id: str, label: str):
        """Submit a label for an item"""
        if user_id not in self.users or item_id not in self.data_items:
            return False
        
        user = self.users[user_id]
        item = self.data_items[item_id]
        
        # Get category from label for binary classification
        category = label if not item.is_binary else item.categories[0]
        
        # Get user's expertise for this category
        expertise = user.get_category_expertise(category)
        
        # Add label to item
        item.add_label(user_id, label, expertise)
        
        # Track that user labeled this item
        if user_id not in self.user_labels:
            self.user_labels[user_id] = []
        self.user_labels[user_id].append(item_id)
        
        return True
    
    def get_user_stats(self, user_id: str) -> Optional[Dict]:
        """Get statistics for a user"""
        if user_id not in self.users:
            return None
        return self.users[user_id].to_dict()
    
    def get_item_status(self, item_id: str) -> Optional[Dict]:
        """Get status of a data item"""
        if item_id not in self.data_items:
            return None
        return self.data_items[item_id].to_dict()

## Example Usage

Demonstrate the labeling system with sample data

In [ ]:
# Initialize the system
system = LabelingSystem()

# Create sample users
user1 = User('user_1', 'Alice', 'alice@example.com')
user2 = User('user_2', 'Bob', 'bob@example.com')
user3 = User('user_3', 'Charlie', 'charlie@example.com')

system.add_user(user1)
system.add_user(user2)
system.add_user(user3)

# Create sample data items
item1 = DataItem('item_1', 'sample_data/tree.jpg', 'image', ['tree', 'bird', 'car'], False)
item2 = DataItem('item_2', 'sample_data/bird.jpg', 'image', ['tree', 'bird', 'car'], False)
item3 = DataItem('item_3', 'sample_data/car.jpg', 'image', ['tree', 'bird', 'car'], False)
item4 = DataItem('item_4', 'sample_data/cat.jpg', 'image', ['animal'], True)  # Binary: is it an animal?

system.add_data_item(item1)
system.add_data_item(item2)
system.add_data_item(item3)
system.add_data_item(item4)

print("System initialized with 3 users and 4 items")

In [ ]:
# Simulate labeling
print("\n=== Labeling Simulation ===")

# User 1 labels items
next_item = system.get_next_item_for_user('user_1')
if next_item:
    print(f"\nUser 1 gets item: {next_item.item_id}")
    system.submit_label('user_1', next_item.item_id, 'tree')
    print(f"User 1 labeled as: tree")

# User 2 labels the same item
system.submit_label('user_2', 'item_1', 'tree')
print(f"User 2 labeled item_1 as: tree")

# User 3 labels differently
system.submit_label('user_3', 'item_1', 'bird')
print(f"User 3 labeled item_1 as: bird")

# Check item status
status = system.get_item_status('item_1')
print(f"\nItem 1 status:")
print(f"  Labels count: {status['labels_count']}")
print(f"  Consensus: {status['consensus_label']}")
print(f"  Confidence: {status['confidence']:.2f}")

In [ ]:
# Check user stats
print("\n=== User Statistics ===")
for user_id in ['user_1', 'user_2', 'user_3']:
    stats = system.get_user_stats(user_id)
    print(f"\n{stats['name']}:")
    print(f"  Total labels: {stats['total_labels']}")
    print(f"  Accuracy: {stats['accuracy']:.2f}")
    print(f"  Expertise by category: {stats['expertise_by_category']}")